In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [2]:
# Step 2: Load the California housing dataset
df = pd.read_excel("housing.csv.xlsx")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


In [3]:
# Step 3: Create income categories for stratified sampling based on median_income
# This ensures all income brackets are proportionally represented in train and test sets
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [4]:
# Step 4: Perform Stratified Shuffle Split (80% train, 20% test)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_index].copy()
    strat_test_set = df.loc[test_index].copy()

# Drop temporary 'income_cat' column so the dataset reverts to its original features
for sett in (strat_train_set, strat_test_set):
    sett.drop("income_cat", axis=1, inplace=True)

In [5]:
# Step 5: Separate predictors and target labels
# We drop 'median_house_value' from predictors and store it separately in housing_labels
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736,INLAND
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373,NEAR OCEAN
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750,INLAND
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264,NEAR OCEAN
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964,<1H OCEAN


In [6]:
# Step 6: Inspect the categorical column 'ocean_proximity'
# NOTE: We use double brackets [['ocean_proximity']] so housing_cat stays as a 2D DataFrame
# Scikit-Learn transformers require a 2D input (DataFrame or 2D NumPy array), not a 1D Series
housing_cat = housing[["ocean_proximity"]]

# Check frequency of each category
print("Category value counts:")
print(housing_cat["ocean_proximity"].value_counts())
housing_cat.head(10)

Category value counts:
ocean_proximity
<1H OCEAN     7277
INLAND        5262
NEAR OCEAN    2124
NEAR BAY      1847
ISLAND           2
Name: count, dtype: int64


,ocean_proximity
12655,INLAND
15502,NEAR OCEAN
2908,INLAND
14053,NEAR OCEAN
20496,<1H OCEAN
1481,NEAR BAY
18125,<1H OCEAN
5830,<1H OCEAN
17989,<1H OCEAN
4861,<1H OCEAN


In [7]:
# Step 7: Method 1 - Ordinal Encoding (OrdinalEncoder)
# Converts each text label into an integer (0, 1, 2, ...)
# Best when categories have a natural ranking (e.g. 'poor' < 'average' < 'good')
# IMPORTANT: Only pass housing_cat (NOT the full housing DataFrame!)
# If you pass all of housing, continuous numbers like longitude/latitude get converted to category ranks!
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

print("First 10 encoded integer values:\n", housing_cat_encoded[:10])
print("\nLearned categories list:", ordinal_encoder.categories_)

First 10 encoded integer values:
 [[1.]
 [4.]
 [1.]
 [4.]
 [0.]
 [3.]
 [0.]
 [0.]
 [0.]
 [0.]]

Learned categories list: [array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
      dtype=object)]


In [8]:
# Step 8: Why Ordinal Encoding is problematic for 'ocean_proximity':
# ML models assume nearby numbers are similar (0 is closer to 1 than to 4).
# Here: category 0 ('<1H OCEAN') and category 4 ('NEAR OCEAN') are geographically similar,
# but the model would think 0 ('<1H OCEAN') and 1 ('INLAND') are closer!
# Let's print the learned integer-to-category mapping:
for code, category in enumerate(ordinal_encoder.categories_[0]):
    print(f"Integer {code} -> {category}")

Integer 0 -> <1H OCEAN
Integer 1 -> INLAND
Integer 2 -> ISLAND
Integer 3 -> NEAR BAY
Integer 4 -> NEAR OCEAN


In [9]:
# Step 9: Method 2 - One-Hot Encoding (OneHotEncoder)
# Preferred for nominal (unordered) categories like 'ocean_proximity'
# Creates one binary (0 or 1) dummy feature per category: 1 when present ('hot'), 0 otherwise ('cold')
cat_encoder = OneHotEncoder()

# fit_transform returns a SciPy CSR sparse matrix by default
# A sparse matrix only stores the memory locations of non-zero values (1s), saving RAM
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16512 stored elements and shape (16512, 5)>

In [10]:
# Step 10: Inspect the One-Hot Encoded output as a dense NumPy array
# Call .toarray() to expand the sparse matrix and see the 0s and 1s across rows and columns
print("Dense binary array (first 5 rows):\n", housing_cat_1hot.toarray()[:5])
print("\nCategories corresponding to the 5 columns:\n", cat_encoder.categories_)

Dense binary array (first 5 rows):
 [[0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]]

Categories corresponding to the 5 columns:
 [array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
      dtype=object)]


In [11]:
# Step 11: Convert the One-Hot Encoded result into a clean Pandas DataFrame
# Use cat_encoder.get_feature_names_out() to generate clear, descriptive column names
# Use housing_cat.index so the row indices match our training set
encoded_columns = cat_encoder.get_feature_names_out(["ocean_proximity"])
print("Generated feature column names:", encoded_columns)

housing_cat_1hot_df = pd.DataFrame(
    housing_cat_1hot.toarray(),
    columns=encoded_columns,
    index=housing_cat.index
)

# Display the final clean one-hot encoded DataFrame
housing_cat_1hot_df.head(10)

Generated feature column names: ['ocean_proximity_<1H OCEAN' 'ocean_proximity_INLAND'
 'ocean_proximity_ISLAND' 'ocean_proximity_NEAR BAY'
 'ocean_proximity_NEAR OCEAN']


,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
12655,0.0,1.0,0.0,0.0,0.0
15502,0.0,0.0,0.0,0.0,1.0
2908,0.0,1.0,0.0,0.0,0.0
14053,0.0,0.0,0.0,0.0,1.0
20496,1.0,0.0,0.0,0.0,0.0
1481,0.0,0.0,0.0,1.0,0.0
18125,1.0,0.0,0.0,0.0,0.0
5830,1.0,0.0,0.0,0.0,0.0
17989,1.0,0.0,0.0,0.0,0.0
4861,1.0,0.0,0.0,0.0,0.0


In [12]:
# Step 12: Looking ahead - Why we use ColumnTransformer in Scikit-Learn
# Right now, we preprocessed numerical features and categorical features separately
# Shape check:
num_features_count = housing.drop("ocean_proximity", axis=1).shape[1]
cat_features_count = housing_cat_1hot_df.shape[1]
print(f"Original numerical features: {num_features_count}")
print(f"One-hot categorical features: {cat_features_count}")
print(f"Total combined features will be: {num_features_count + cat_features_count}")
# In the next lesson, we will combine SimpleImputer, StandardScaler, and OneHotEncoder
# into a single automated ColumnTransformer & Pipeline!

Original numerical features: 8
One-hot categorical features: 5
Total combined features will be: 13
